# 岩石力学分析

##  岩体抗力算例计算

##  计算支护结构抗力

In [82]:
import math
import numpy as np
from scipy.optimize import brentq
def hoek_brown_params(GSI, mi, D):
    '''
    Hoek_Brown 经验准则
    支护挤压岩体，岩体抵抗变形所能提供的侧向抗力
    GSI:地质强度指标
    mi: 完整岩石
    D :扰动系数 0~1
    '''
    mb = mi* np.exp((GSI -100)/(28 - 14*D))
    s = np.exp((GSI -100)/(9-3*D))
    a = 0.5 + (1/6) * (np.exp(-GSI/15))-np.exp(-20/3)

    return mb, s, a

def calc_rock_mass_resistence(sigma_ci, GSI, mi, D, sigma1):
    '''
    已知轴向应力sigma1 ,求解对应侧向应抗力 sigma3（Mpa）
    非线性方程，使用二分法经行求解
    '''
    
    mb, s, a = hoek_brown_params(GSI,  mi, D)
    sigma_cmass = mb*sigma_ci*(s)**a
     
    def f(s3, sigma_ci, GSI, mi, D, sigma1):
            return s3 + sigma_ci *mb*((s3 / sigma_ci + s)**a) - sigma1 
    if sigma1>sigma_cmass:
        f_bound = lambda s3:f(s3, sigma_ci, GSI, mi, D, sigma1)
        s3_res = brentq(f_bound , 0, sigma1*5)

    else:
        s3_res = 0
        
       
        
             
    return s3_res, mb, s, a
        
        

        
    

In [84]:
sigma_ci = 45  # 单块岩石单轴抗压强度（Mpa）
GSI = 55       # 岩体ＧＳＩ
mi = 12.0
sigma1 = 15     # 无侧限轴向大主应力（Mpa）
D = 0
s3_res, mb, s, a = calc_rock_mass_resistence(sigma_ci, GSI, mi, D, sigma1) 

print(f"mb = {mb:.3f}, s = {s:.5f}, a= {a:.4f}")
print(f" 对应岩体极限侧向抗力 sigma3 = {s3_res:.3f}Mpa")


mb = 2.406, s = 0.00674, a= 0.5030
 对应岩体极限侧向抗力 sigma3 = 0.521Mpa


In [79]:
# 函数调试
sigma_ci = 45  # 单块岩石单轴抗压强度（Mpa）
GSI = 55       # 岩体ＧＳＩ
mi = 12.0
sigma1 = 10     # 无侧限轴向大主应力（Mpa）
D = 0
mb = mi* np.exp((GSI -100)/(28 - 14*D))
s = np.exp((GSI -100)/(9-3*D))
a = 0.5 + (1/6) * (np.exp(-GSI/15))-np.exp(-20/3)

def f(s3, sigma_ci, GSI, mi, D, sigma1):
    return  s3 + sigma_ci *mb*((s3 / sigma_ci + s)**a) - sigma1 
    

print(f"mb = {mb:.3f}, s = {s:.5f}, a= {a:.4f}")
sigma_cmass = mb*sigma_ci*(s)**a

 
if sigma1>sigma_cmass:
    f_bound = lambda s3:f(s3, sigma_ci, GSI, mi, D, sigma1)
    s3_res = brentq(f_bound , 0, sigma1*5) 

else:
    s3_res = 0
    
print(f" 对应岩体极限侧向抗力 sigma3 = {s3_res:.3f}Mpa")

mb = 2.406, s = 0.00674, a= 0.5030
 对应岩体极限侧向抗力 sigma3 = 0.085Mpa


## 给出应力张量计算主应力大小及方向

###  无依赖封装函数

In [14]:
import numpy as np

def principle_stress(sigma_x, sigma_y, sigma_z, tau_xy, tau_yz, tau_xz, compressoin_positive = True ):

    '''
    参数输入次序：
    σ_x = 18
    σ_y = 12
    σ_z = 8

    τ_xy = 3
    τ_yz = 1.2
    τ_xz = 2.1
    给出应力张量，计算主应力大小及方向向量
    补充函数，直接输入张量矩阵经行运算
    其次计算破裂面方向向量（叉积向量）
    '''
    if compressoin_positive:
        sig_mat  = -np.array([
            [sigma_x, tau_xy, tau_xz],
            [tau_xy, sigma_y, tau_yz],
            [tau_xz, tau_yz, sigma_z]
        ])
    else:
        sig_mat = np.array([
             [sigma_x, tau_xy, tau_xz],
             [tau_xy, sigma_y, tau_yz],
             [tau_xz, tau_yz, sigma_z]
         ])

    s1 = eigvals_comp[sort_idx[0]]
    s2 = eigvals_comp[sort_idx[1]]
    s3 = eigvals_comp[sort_idx[2]]

    vec1 = eigvecs[:, sort_idx[0]]
    vec2 = eigvecs[:, sort_idx[1]]
    vec3 = eigvecs[:, sort_idx[2]]


    print(f"σ1 = {s1} ")
    print(f"σ2 = {s2} ")
    print(f"σ3 = {s3} ")
    print(f"σ1 主方向向量{vec1.round(3)}")
    return s1, s2, s3, vec1, vec2, vec3

    



In [15]:
σ_x = 18
σ_y = 12
σ_z = 8

τ_xy = 3
τ_yz = 1.2
τ_xz = 2.1

principle_stress(σ_x, σ_y, σ_z, τ_xy, τ_yz, τ_xz)


σ1 = 19.733708980451674 
σ2 = 10.784211236725238 
σ3 = 7.482079782823087 
σ1 主方向向量[-0.902 -0.381 -0.201]


(np.float64(19.733708980451674),
 np.float64(10.784211236725238),
 np.float64(7.482079782823087),
 array([-0.90248887, -0.38119772, -0.20050471]),
 array([-0.40406638,  0.910519  ,  0.08766698]),
 array([-0.14914489, -0.16013569,  0.97576245]))

In [11]:
compressoin_positive= True

sigma_x = 18
sigma_y = 12
sigma_z = 8

tau_xy = 3
tau_yz = 1.2
tau_xz = 2.1
if compressoin_positive:
     sig_mat  = -np.array([
         [sigma_x, tau_xy, tau_xz],
         [tau_xy, sigma_y, tau_yz],
         [tau_xz, tau_yz, sigma_z]
     ])
else:
    sig_mat = np.array([
         [sigma_x, tau_xy, tau_xz],
         [tau_xy, sigma_y, tau_yz],
         [tau_xz, tau_yz, sigma_z]
     ])


eigvals, eigvecs = np.linalg.eig(sig_mat)
eigvals_comp = -eigvals

sort_idx = np.argsort(-eigvals_comp)

s1 = eigvals_comp[sort_idx[0]]
s2 = eigvals_comp[sort_idx[1]]
s3 = eigvals_comp[sort_idx[2]]

vec1 = eigvecs[:, sort_idx[0]]
vec2 = eigvecs[:, sort_idx[1]]
vec3 = eigvecs[:, sort_idx[2]]

print(f"σ1 = {s1} ")
print(f"σ2 = {s2} ")
print(f"σ3 = {s3} ")


print(f"σ1 主方向向量{vec1.round(3)}")



σ1 = 19.733708980451674 
σ2 = 10.784211236725238 
σ3 = 7.482079782823087 
σ1 主方向向量[-0.902 -0.381 -0.201]


### 依赖 pyvista ，对图像文件处理，计算主应力

In [18]:
import pyvista as pv

def vtu_calc_principal(vtu_file):
    mesh = pv.read(vtu_file)
    sigma_all = mesh["Sigma"]  # N_points × 6
    npts = sigma_all.shape[0]

    s1_arr = np.zeros(npts)
    s2_arr = np.zeros(npts)
    s3_arr = np.zeros(npts)

    for i in range(npts):
        σ_xx, σ_yy, σ_zz, τ_xy, τ_yz, τ_xz = sigma_all[i, :]
        s1, s2, s3, _, _, _ = principle_stress(σ_xx, σ_yy, σ_zz, τ_xy, τ_yz, τ_xz, compressoin_positive= False)

        s1_arr[i] = s1
        s2_arr[i] = s2
        s3_arr[i] = s3

    mesh["sigma1"] = s1_arr
    mesh["sigma2"] = s2_arr
    mesh["sigma3"] = s3_arr

    return mesh
        
        





## 隧道开挖应力计算

### 隧道开挖后应力计算方法

In [ ]:
import math

def kirsch_circle_tunnel(r, a, p0):
    """
    圆形隧洞Kirsch弹性解，静水地应力
    :param r: 离开挖中心距离 m
    :param a: 隧道半径 m
    :param p0: 初始静水地应力 MPa
    :return: sigma_r, sigma_theta, sigma_m, J2, sigma_e
    """
    if r <= a:
        raise ValueError("r必须大于洞半径a，此公式为洞外围岩")
    ratio = (a / r) ** 2
    sigma_r = p0 * (1 - ratio)
    sigma_theta = p0 * (1 + ratio)

    # 应力球张量：平均应力
    sigma_m = (sigma_r + sigma_theta) / 2

    # 应力偏张量分量
    s_r = sigma_r - sigma_m
    s_t = sigma_theta - sigma_m

    # 偏不变量 J2
    J2 = 0.5 * (s_r**2 + s_t**2)
    sigma_e = math.sqrt(3 * J2)
    return sigma_r, sigma_theta, sigma_m, J2, sigma_e

# ========== 输入参数【可自行修改】==========
a = 3.0    # 隧道半径，m
p0 = 10.0  # 初始地应力，MPa
r = 5.0    # 离开挖中心距离 m

sigma_r, sigma_theta, sigma_m, J2, sigma_e = kirsch_circle_tunnel(r, a, p0)

print("===== 圆形隧洞开挖应力计算(Kirsch解) =====")
print(f"隧道半径 a = {a} m")
print(f"离开挖中心距离 r = {r} m")
print(f"初始静水地应力 p0 = {p0} MPa")
print(f"径向应力 σ_r = {sigma_r:.3f} MPa")
print(f"环向应力 σ_θ = {sigma_theta:.3f} MPa")
print(f"平均应力(球张量) σ_m = {sigma_m:.3f} MPa")
print(f"偏应力第二不变量 J2 = {J2:.3f} MPa²")
print(f"Mises等效应力 σ_e = {sigma_e:.3f} MPa")


### 对开挖后远处一点应力计算后旋转

In [ ]:
import numpy as np
import math

def rotate_stress_tensor(sigma, theta_deg):
    """
    二阶平面应力张量旋转
    :param sigma: 2x2原始应力张量 numpy array
    :param theta_deg: 旋转角度，逆时针，单位°
    :return: sigma_rot 旋转后应力张量
    """
    theta = math.radians(theta_deg)
    c = math.cos(theta)
    s = math.sin(theta)
    R = np.array([[c, s],
                  [-s, c]])
    R_T = R.T
    sigma_rot = R @ sigma @ R_T
    return sigma_rot

def decompose_spherical_dev(sigma):
    """
    应力张量分解：球张量 + 偏张量
    :param sigma: 2x2应力张量
    :return: sigma_m(平均应力), sigma_sph(球张量), s(偏张量), J2(偏第二不变量)
    """
    sigma_m = (sigma[0,0] + sigma[1,1]) / 2.0
    sigma_sph = sigma_m * np.eye(2)
    s = sigma - sigma_sph
    J2 = 0.5 * np.trace(s @ s.T)
    return sigma_m, sigma_sph, s, J2

def kirsch_polar(r, a, p0):
    """Kirsch静水地应力极坐标应力（无剪应力 tau_rθ=0）"""
    ratio = (a / r)**2
    sigma_r = p0 * (1 - ratio)
    sigma_theta = p0 * (1 + ratio)
    tau_rtheta = 0.0
    sigma_polar = np.array([[sigma_r, tau_rtheta],
                            [tau_rtheta, sigma_theta]])
    return sigma_polar

# ====================== 算例输入参数 ======================
a = 3.0       # 隧道半径 m
r = 5.0       # 离开挖中心距离 m
p0 = 10.0     # 初始静水地应力 MPa
theta = 45.0  # 旋转角度(°)，极坐标→直角坐标逆时针旋转

# 1.得到极坐标应力张量
sigma_polar = kirsch_polar(r, a, p0)
# 2.张量旋转
sigma_cart = rotate_stress_tensor(sigma_polar, theta)

# 3.分别分解球张量、偏张量
sigma_m_p, sph_p, s_p, J2_p = decompose_spherical_dev(sigma_polar)
sigma_m_c, sph_c, s_c, J2_c = decompose_spherical_dev(sigma_cart)

# =========输出结果=========
print("===== 张量旋转算例（Kirsch圆形隧洞） =====")
print(f"隧道半径 a = {a} m, 测点r = {r} m, p0 = {p0} MPa")
print(f"旋转角度 θ = {theta} °")
print("\n【极坐标原始应力张量】")
print(np.round(sigma_polar,4))
print(f"极坐标平均应力 σ_m = {sigma_m_p:.4f} MPa")
print("极坐标球张量：")
print(np.round(sph_p,4))
print("极坐标偏张量：")
print(np.round(s_p,4))
print(f"极坐标 J2 = {J2_p:.4f} MPa²")

print("\n【旋转后笛卡尔应力张量】")
print(np.round(sigma_cart,4))
print(f"笛卡尔平均应力 σ_m = {sigma_m_c:.4f} MPa")
print("笛卡尔球张量：")
print(np.round(sph_c,4))
print("笛卡尔偏张量：")
print(np.round(s_c,4))
print(f"笛卡尔 J2 = {J2_c:.4f} MPa²")

print("\n===== 不变量校验（旋转前后应当相等） =====")
print(f"I1原始 = {np.trace(sigma_polar):.4f}, I1旋转后 = {np.trace(sigma_cart):.4f}")
print(f"J2原始 = {J2_p:.4f}, J2旋转后 = {J2_c:.4f}")


## 计算判别岩体破坏条件

### mohr_coulomb 破坏准则(受剪破坏) & griffith （裂缝受拉破坏）

In [1]:
import numpy as np
import math

def principal_stress(sigma_tensor):
    """
    求解应力张量主应力，降序 σ1 ≥ σ2 ≥ σ3（压为正）
    sigma_tensor: 2×2 或 3×3 numpy数组
    """
    eigvals, eigvecs = np.linalg.eig(sigma_tensor)
    # 去除微小虚部（数值噪声）
    eigvals = np.real(eigvals)
    eig_sorted = np.sort(eigvals)[::-1]
    return eig_sorted

def lode_angle(sigma_tensor):
    """计算罗德角 θ_l(°)，输出J2,J3,主应力"""
    eigvals, _ = np.linalg.eig(sigma_tensor)
    eigvals = np.real(eigvals)
    s1, s2, s3 = np.sort(eigvals)[::-1]
    sigma_m = (s1 + s2 + s3)/3
    # 偏应力分量
    s_1 = s1 - sigma_m
    s_2 = s2 - sigma_m
    s_3 = s3 - sigma_m
    J2 = 0.5*(s_1**2 + s_2**2 + s_3**2)
    J3 = s_1 * s_2 * s_3
    if J2 < 1e-12:
        return 0.0, J2, J3, s1, s2, s3
    cos3l = (3*math.sqrt(3)/2) * J3 / (J2**(1.5))
    cos3l = np.clip(cos3l, -1.0, 1.0)
    theta_l_rad = math.acos(cos3l)/3
    theta_l_deg = math.degrees(theta_l_rad)
    return theta_l_deg, J2, J3, s1, s2, s3

def mohr_coulomb_judge(sigma1, sigma3, c, phi_deg):
    """
    Mohr-Coulomb 剪切破坏判定
    :param sigma1:最大主应力
    :param sigma3:最小主应力
    :param c:黏聚力 MPa
    :param phi_deg:内摩擦角 °
    :return: FS, sigma1_f, is_shear_fail, msg
    """
    phi = math.radians(phi_deg)
    sinφ = math.sin(phi)
    cosφ = math.cos(phi)
    sigma1_f = sigma3 * (1+sinφ)/(1-sinφ) + 2*c*cosφ/(1-sinφ)
    FS = sigma1_f / sigma1
    if FS < 1.0:
        is_shear_fail = True
        msg = "MC准则：发生剪切破坏"
    else:
        is_shear_fail = False
        msg = "MC准则：单元稳定，无剪切破坏"
    return FS, sigma1_f, is_shear_fail, msg

def griffith_judge(sigma1, sigma3, T0):
    """
    Griffith脆性断裂准则（Inglis岩石形式）
    T0:单轴抗拉强度(正值 MPa)
    return: is_brittle_fail, msg
    """
    if sigma1 + 3*sigma3 > 0:
        # 压剪断裂分支
        left = (sigma1 - sigma3)**2
        right = 8 * T0 * (sigma1 + sigma3)
        if left >= right:
            return True, "Griffith准则：压剪脆性断裂"
        else:
            return False, "Griffith准则：无裂纹扩展，稳定"
    else:
        # 纯张拉断裂分支
        if sigma3 <= -T0:
            return True, "Griffith准则：纯张拉断裂"
        else:
            return False, "Griffith准则：无裂纹扩展，稳定"

def element_failure_analysis(sigma_tensor, c, phi_deg, T0):
    """
    总入口函数：输入应力张量，一次性全套计算
    返回所有结果
    """
    theta_l, J2, J3, sigma1, sigma2, sigma3 = lode_angle(sigma_tensor)
    # MC计算
    FS_mc, sigma1_f_mc, shear_fail, msg_mc = mohr_coulomb_judge(sigma1, sigma3, c, phi_deg)
    # Griffith计算
    brittle_fail, msg_griff = griffith_judge(sigma1, sigma3, T0)

    # 综合判定：任意一种破坏发生即判定单元失效
    element_fail = shear_fail or brittle_fail
    all_msg = []
    if shear_fail:
        all_msg.append(msg_mc)
    if brittle_fail:
        all_msg.append(msg_griff)
    if not shear_fail and not brittle_fail:
        all_msg.append("单元安全，无剪切、无脆性断裂")
    return {
        "sigma1":sigma1, "sigma2":sigma2, "sigma3":sigma3,
        "theta_l":theta_l, "J2":J2, "J3":J3,
        "FS_mc":FS_mc, "sigma1_f_mc":sigma1_f_mc,
        "shear_fail":shear_fail, "msg_mc":msg_mc,
        "brittle_fail":brittle_fail, "msg_griff":msg_griff,
        "element_fail":element_fail,
        "summary":"；".join(all_msg)
    }

# ======================【算例1：隧洞r=5m单元（复用前面Kirsch旋转后张量）】======================
if __name__ == "__main__":
    # 应力张量 2D -> 扩展为3×3张量用于罗德角计算
    sigma = np.array([[10.0, 3.6, 0.0],
                      [3.6, 10.0, 0.0],
                      [0.0, 0.0, 10.0]])
    # 岩石材料参数
    c = 2.2         # 黏聚力 MPa
    phi_deg = 32    # 内摩擦角 °
    T0 = 2.0        # 岩石单轴抗拉强度 MPa

    res = element_failure_analysis(sigma, c, phi_deg, T0)

    print("="*60)
    print("===== 单元破坏综合判定：Mohr-Coulomb + Griffith =====")
    print("输入应力张量：")
    print(sigma)
    print(f"\n主应力 σ1={res['sigma1']:.3f}, σ2={res['sigma2']:.3f}, σ3={res['sigma3']:.3f} MPa")
    print(f"罗德角 θ_l = {res['theta_l']:.2f} °")
    print(f"偏不变量 J2={res['J2']:.3f} MPa², J3={res['J3']:.3f} MPa³")
    print("\n【Mohr-Coulomb剪切判据】")
    print(f"临界破坏σ1_f = {res['sigma1_f_mc']:.3f} MPa；安全系数FS = {res['FS_mc']:.3f}")
    print(res["msg_mc"])
    print("\n【Griffith脆性断裂判据】")
    print(res["msg_griff"])
    print("\n【综合判定】")
    print(res["summary"])
    if res["element_fail"]:
        print("⚠️ 单元发生破坏！")
    else:
        print("✅ 单元保持完整，未发生破坏")
    print("="*60)

    # ==========算例2：洞壁单元，r=a=3m，σr=0,σθ=20MPa，测试==========
    print("\n\n===== 算例2：隧道洞壁单元 ======")
    sigma_wall = np.array([[20,0,0],
                           [0,0,0],
                           [0,0,0]])
    res2 = element_failure_analysis(sigma_wall, c, phi_deg, T0)
    print(f"σ1={res2['sigma1']:.3f}, σ2={res2['sigma2']:.3f}, σ3={res2['sigma3']:.3f}")
    print(f"罗德角={res2['theta_l']:.2f}°")
    print(f"MC安全系数FS={res2['FS_mc']:.3f}")
    print(res2["summary"])


===== 单元破坏综合判定：Mohr-Coulomb + Griffith =====
输入应力张量：
[[10.   3.6  0. ]
 [ 3.6 10.   0. ]
 [ 0.   0.  10. ]]

主应力 σ1=13.600, σ2=10.000, σ3=6.400 MPa
罗德角 θ_l = 30.00 °
偏不变量 J2=12.960 MPa², J3=-0.000 MPa³

【Mohr-Coulomb剪切判据】
临界破坏σ1_f = 28.767 MPa；安全系数FS = 2.115
MC准则：单元稳定，无剪切破坏

【Griffith脆性断裂判据】
Griffith准则：无裂纹扩展，稳定

【综合判定】
单元安全，无剪切、无脆性断裂
✅ 单元保持完整，未发生破坏


===== 算例2：隧道洞壁单元 ======
σ1=20.000, σ2=0.000, σ3=0.000
罗德角=0.00°
MC安全系数FS=0.397
MC准则：发生剪切破坏；Griffith准则：压剪脆性断裂
